# Steam Community Market Scraper for CS2 Items

This notebook collects and structures Steam Community Market data for two types of CS2 items:

- **Indistinguishable items**, such as cases, where all listings are effectively interchangeable.
- **Distinguishable items**, such as weapon skins, where each listing can have its own float value, pattern template and stickers.

The project uses different tools depending on the structure of the data:
- `requests`, `BeautifulSoup` and regular expressions for static aggregate market data.
- `Selenium` for rendered listing pages of distinguishable items.
- `Scrapy` as a scaling layer for processing multiple item pages.

## Legal and technical scope

The scraper only collects publicly visible market data. It does not log into a Steam account, does not place buy orders, does not buy or sell items, and does not interact with marketplace functionality beyond reading public pages.

Request delays are used to reduce server load. The number of pages can be changed with one configuration variable.

In [192]:
import json
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Union
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [193]:
APP_ID = 730
COUNTRY = "PL"
LANGUAGE = "english"
CURRENCY = 1  # USD in Steam Community Market endpoints

INDISTINGUISHABLE_ITEM = "Revolution Case"
DISTINGUISHABLE_ITEM = "AK-47 | Redline (Field-Tested)"

FLOAT_THRESHOLD = 0.15

# For all available pages, set SELENIUM_PAGES_TO_SCRAPE = "max".
# For quicker runs, set an integer such as 3.
SELENIUM_RESULTS_PER_PAGE = 100
SELENIUM_PAGES_TO_SCRAPE = 3

REQUEST_DELAY_SECONDS = 0.7
SELENIUM_PAGE_DELAY_SECONDS = 1.0

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## Helper functions

In [194]:
def build_listing_url(market_hash_name: str, app_id: int = APP_ID) -> str:
    return f"https://steamcommunity.com/market/listings/{app_id}/{quote(market_hash_name)}"


def make_request(url: str, params: Optional[dict] = None) -> requests.Response:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9"
    }
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY_SECONDS)
    return response


def clean_price(value: Optional[str]) -> Optional[float]:
    if value is None:
        return None
    match = re.search(r"[0-9]+(?:[.,][0-9]+)?", value.replace(",", ""))
    return float(match.group(0)) if match else None


def extract_item_nameid(html: str) -> Optional[int]:
    match = re.search(r"Market_LoadOrderSpread\(\s*(\d+)\s*\)", html)
    return int(match.group(1)) if match else None


def extract_balanced_json_object(text: str, variable_name: str) -> Optional[dict]:
    prefix = f"var {variable_name} ="
    start = text.find(prefix)
    if start == -1:
        return None

    brace_start = text.find("{", start)
    if brace_start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for i in range(brace_start, len(text)):
        ch = text[i]

        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                raw = text[brace_start:i + 1]
                return json.loads(raw)

    return None

# Part 1: Indistinguishable item pipeline

For cases and other interchangeable market items, the relevant data is aggregate data:
- current market overview,
- historical price and volume,
- buy and sell order distribution.

This part uses `requests`, `BeautifulSoup` and regular expressions because the required identifiers and aggregate data can be obtained without browser interaction.

In [195]:
def get_market_page_html(market_hash_name: str) -> str:
    url = build_listing_url(market_hash_name)
    return make_request(url).text


def scrape_item_overview(market_hash_name: str) -> pd.DataFrame:
    html = get_market_page_html(market_hash_name)
    soup = BeautifulSoup(html, "html.parser")

    title = soup.find("title").get_text(strip=True) if soup.find("title") else None
    item_nameid = extract_item_nameid(html)

    return pd.DataFrame([{
        "market_hash_name": market_hash_name,
        "listing_url": build_listing_url(market_hash_name),
        "page_title": title,
        "item_nameid": item_nameid,
        "scraped_at": datetime.now()
    }])


def parse_steam_history_date(date_text):
    clean = re.sub(r":\s*\+0$", ":00 +0000", date_text)
    return pd.to_datetime(clean, errors="coerce")


def scrape_price_history(market_hash_name: str) -> pd.DataFrame:
    html = get_market_page_html(market_hash_name)

    match = re.search(r"var\s+line1\s*=\s*(\[.*?\]);", html, flags=re.DOTALL)
    if not match:
        return pd.DataFrame(columns=["market_hash_name", "date", "median_price", "volume"])

    raw_prices = json.loads(match.group(1))

    rows = []
    for point in raw_prices:
        rows.append({
            "market_hash_name": market_hash_name,
            "date": parse_steam_history_date(point[0]),
            "median_price": float(point[1]),
            "volume": int(str(point[2]).replace(",", ""))
        })

    return pd.DataFrame(rows)


def scrape_order_histogram(item_nameid: int, market_hash_name: str) -> pd.DataFrame:
    url = "https://steamcommunity.com/market/itemordershistogram"
    params = {
        "country": COUNTRY,
        "language": LANGUAGE,
        "currency": CURRENCY,
        "item_nameid": item_nameid,
        "two_factor": 0
    }

    data = make_request(url, params=params).json()
    rows = []

    for side, graph_key in [("buy", "buy_order_graph"), ("sell", "sell_order_graph")]:
        for point in data.get(graph_key, []):
            rows.append({
                "market_hash_name": market_hash_name,
                "side": side,
                "price": float(point[0]),
                "quantity": int(point[1]),
                "description": point[2] if len(point) > 2 else None,
                "scraped_at": datetime.now()
            })

    return pd.DataFrame(rows)


def scrape_indistinguishable_item(market_hash_name: str) -> Dict[str, pd.DataFrame]:
    overview_df = scrape_item_overview(market_hash_name)
    item_nameid = overview_df.loc[0, "item_nameid"]

    history_df = scrape_price_history(market_hash_name)

    if pd.notna(item_nameid):
        orders_df = scrape_order_histogram(int(item_nameid), market_hash_name)
    else:
        orders_df = pd.DataFrame()

    return {
        "overview": overview_df,
        "history": history_df,
        "orders": orders_df
    }

In [196]:
case_result = scrape_indistinguishable_item(INDISTINGUISHABLE_ITEM)

case_overview_df = case_result["overview"]
case_history_df = case_result["history"]
case_orders_df = case_result["orders"]

case_overview_df.head()

,market_hash_name,listing_url,page_title,item_nameid,scraped_at
0,Revolution Case,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for Revolut...,176358765,2026-05-12 01:43:01.054404


In [197]:
case_history_df.head()

,market_hash_name,date,median_price,volume
0,Revolution Case,2023-02-10 01:00:00+00:00,13.462,79764
1,Revolution Case,2023-02-11 01:00:00+00:00,8.041,79886
2,Revolution Case,2023-02-12 01:00:00+00:00,6.889,78300
3,Revolution Case,2023-02-13 01:00:00+00:00,6.960,58451
4,Revolution Case,2023-02-14 01:00:00+00:00,5.986,57769


In [198]:
case_orders_df.head()

,market_hash_name,side,price,quantity,description,scraped_at
0,Revolution Case,buy,0.46,1373,"1,373 buy orders at $0.46 or higher",2026-05-12 01:43:04.532869
1,Revolution Case,buy,0.43,16135,"16,135 buy orders at $0.43 or higher",2026-05-12 01:43:04.532869
2,Revolution Case,buy,0.42,22956,"22,956 buy orders at $0.42 or higher",2026-05-12 01:43:04.532869
3,Revolution Case,buy,0.41,56577,"56,577 buy orders at $0.41 or higher",2026-05-12 01:43:04.532869
4,Revolution Case,buy,0.40,81316,"81,316 buy orders at $0.40 or higher",2026-05-12 01:43:04.532869


## Feature engineering for indistinguishable items

The historical dataset is extended with relative changes in median price and volume. The order book data is extended with best buy, best sell and spread values.

In [199]:
def engineer_case_features(price_history_df: pd.DataFrame, order_histogram_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    history = price_history_df.copy()

    if not history.empty:
        history = history.sort_values("date")
        history["median_price_change_pct"] = history["median_price"].pct_change()
        history["volume_change_pct"] = history["volume"].pct_change()
        history["median_price_7d_avg"] = history["median_price"].rolling(7, min_periods=1).mean()
        history["volume_7d_avg"] = history["volume"].rolling(7, min_periods=1).mean()

    orders = order_histogram_df.copy()

    if not orders.empty:
        best_buy_price = orders.loc[orders["side"].eq("buy"), "price"].max()
        best_sell_price = orders.loc[orders["side"].eq("sell"), "price"].min()

        spread_abs = best_sell_price - best_buy_price if pd.notna(best_buy_price) and pd.notna(best_sell_price) else np.nan
        spread_pct = spread_abs / best_sell_price if pd.notna(best_sell_price) and best_sell_price != 0 else np.nan

        orders["best_buy_price"] = best_buy_price
        orders["best_sell_price"] = best_sell_price
        orders["buy_sell_spread_abs"] = spread_abs
        orders["buy_sell_spread_pct"] = spread_pct

    return {"history": history, "orders": orders}


case_features = engineer_case_features(case_history_df, case_orders_df)
case_history_features_df = case_features["history"]
case_orders_features_df = case_features["orders"]

case_history_features_df.head()

,market_hash_name,date,median_price,volume,median_price_change_pct,volume_change_pct,median_price_7d_avg,volume_7d_avg
0,Revolution Case,2023-02-10 01:00:00+00:00,13.462,79764,NaN,NaN,13.4620,79764.000000
1,Revolution Case,2023-02-11 01:00:00+00:00,8.041,79886,-0.402689,0.001530,10.7515,79825.000000
2,Revolution Case,2023-02-12 01:00:00+00:00,6.889,78300,-0.143266,-0.019853,9.4640,79316.666667
3,Revolution Case,2023-02-13 01:00:00+00:00,6.960,58451,0.010306,-0.253499,8.8380,74100.250000
4,Revolution Case,2023-02-14 01:00:00+00:00,5.986,57769,-0.139943,-0.011668,8.2676,70834.000000


In [200]:
case_orders_features_df.head()

,market_hash_name,side,price,quantity,description,scraped_at,best_buy_price,best_sell_price,buy_sell_spread_abs,buy_sell_spread_pct
0,Revolution Case,buy,0.46,1373,"1,373 buy orders at $0.46 or higher",2026-05-12 01:43:04.532869,0.46,0.49,0.03,0.061224
1,Revolution Case,buy,0.43,16135,"16,135 buy orders at $0.43 or higher",2026-05-12 01:43:04.532869,0.46,0.49,0.03,0.061224
2,Revolution Case,buy,0.42,22956,"22,956 buy orders at $0.42 or higher",2026-05-12 01:43:04.532869,0.46,0.49,0.03,0.061224
3,Revolution Case,buy,0.41,56577,"56,577 buy orders at $0.41 or higher",2026-05-12 01:43:04.532869,0.46,0.49,0.03,0.061224
4,Revolution Case,buy,0.40,81316,"81,316 buy orders at $0.40 or higher",2026-05-12 01:43:04.532869,0.46,0.49,0.03,0.061224


# Part 2: Distinguishable item pipeline

For skins, individual listings matter. The page contains listing rows and embedded JavaScript objects with asset data. These objects include important item-specific attributes such as pattern template, wear rating and inspect data.

Selenium is used to load listing pages and pagination, while the final extraction is done by parsing the rendered HTML and embedded JavaScript data.

In [201]:
def create_chrome_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if headless:
        options.add_argument("--headless=new")
    return webdriver.Chrome(options=options)


def get_total_listings_from_html(html: str) -> Optional[int]:
    soup = BeautifulSoup(html, "html.parser")
    total = soup.select_one("#searchResults_total")
    if total:
        text = total.get_text(strip=True).replace(",", "")
        if text.isdigit():
            return int(text)

    match = re.search(r'"total_count"\s*:\s*(\d+)', html)
    return int(match.group(1)) if match else None


def parse_stickers_from_asset(asset: dict) -> List[Optional[str]]:
    stickers = []

    for desc in asset.get("descriptions", []):
        value = desc.get("value", "")
        if "Sticker:" not in value:
            continue

        text = BeautifulSoup(value, "html.parser").get_text(" ")
        match = re.search(r"Sticker:\s*(.+)", text)
        if match:
            stickers = [x.strip() for x in match.group(1).split(",") if x.strip()]
            break

    stickers = stickers[:4]
    while len(stickers) < 4:
        stickers.append(None)

    return stickers


def parse_asset_properties(asset: dict) -> dict:
    output = {
        "pattern_template": None,
        "float_value": None,
        "inspect_data": None
    }

    for prop in asset.get("asset_properties", []):
        property_id = str(prop.get("propertyid"))

        if property_id == "1":
            output["pattern_template"] = int(prop["int_value"]) if prop.get("int_value") is not None else None
        elif property_id == "2":
            output["float_value"] = float(prop["float_value"]) if prop.get("float_value") is not None else None
        elif property_id == "6":
            output["inspect_data"] = prop.get("string_value")

    return output


def build_inspect_link(asset: dict, inspect_data: Optional[str]) -> Optional[str]:
    actions = asset.get("market_actions") or asset.get("actions") or []
    if not actions or not inspect_data:
        return None

    link_template = actions[0].get("link")
    if not link_template:
        return None

    return link_template.replace("%propid:6%", inspect_data)

In [202]:
def parse_sticker_name_from_src(src):
    if not src:
        return None

    filename = src.split("/")[-1]
    name = filename.split(".")[0]
    return name.replace("_", " ")


def extract_asset_data_from_assets(assets, asset_id):
    float_value = np.nan
    pattern_template = np.nan
    inspect_link = None

    asset = (
        assets
        .get(str(APP_ID), {})
        .get("2", {})
        .get(str(asset_id), {})
    )

    for prop in asset.get("asset_properties", []):
        if str(prop.get("propertyid")) == "1":
            pattern_template = prop.get("int_value")
        elif str(prop.get("propertyid")) == "2":
            float_value = prop.get("float_value")
        elif str(prop.get("propertyid")) == "6":
            inspect_link = prop.get("string_value")

    if pd.notna(float_value):
        float_value = float(float_value)

    if pd.notna(pattern_template):
        pattern_template = int(pattern_template)

    return float_value, pattern_template, inspect_link

def build_listing_url(market_hash_name):
    return f"https://steamcommunity.com/market/listings/{APP_ID}/{quote(market_hash_name)}"


def build_listing_anchor_url(market_hash_name, listing_id):
    return f"{build_listing_url(market_hash_name)}#listing_{listing_id}"


def parse_steam_listing_page(html, market_hash_name):
    soup = BeautifulSoup(html, "html.parser")
    assets = extract_balanced_json_object(html, "g_rgAssets") or {}

    rows = []

    for row in soup.select(".market_recent_listing_row"):
        row_id = row.get("id", "")
        listing_match = re.search(r"listing_(\d+)", row_id)
        listing_id = listing_match.group(1) if listing_match else None

        row_text = str(row)
        asset_match = re.search(
            r"BuyMarketListing\('listing',\s*'(\d+)',\s*\d+,\s*'2',\s*'(\d+)'\)",
            row_text
        )
        asset_id = asset_match.group(2) if asset_match else None

        price_with_fee = row.select_one(".market_listing_price_with_fee")
        price_without_fee = row.select_one(".market_listing_price_without_fee")

        price_with_fee = clean_price(price_with_fee.get_text(" ", strip=True)) if price_with_fee else np.nan
        price_without_fee = clean_price(price_without_fee.get_text(" ", strip=True)) if price_without_fee else np.nan

        inspect_a = row.select_one(".market_listing_row_action a")
        inspect_link_html = inspect_a.get("href") if inspect_a else None

        stickers = [
            parse_sticker_name_from_src(img.get("src"))
            for img in row.select(".sticker_info img")
        ]

        stickers = stickers[:4]
        while len(stickers) < 4:
            stickers.append(None)

        float_value, pattern_template, inspect_link_asset = extract_asset_data_from_assets(
            assets,
            asset_id
        )

        rows.append({
            "market_hash_name": market_hash_name,
            "listing_id": listing_id,
            "asset_id": asset_id,
            "listing_url": build_listing_anchor_url(market_hash_name, listing_id),
            "price_with_fee": price_with_fee,
            "price_without_fee": price_without_fee,
            "total_price": price_with_fee,
            "float": float_value,
            "pattern_template": pattern_template,
            "inspect_link": inspect_link_asset or inspect_link_html,
            "sticker_1": stickers[0],
            "sticker_2": stickers[1],
            "sticker_3": stickers[2],
            "sticker_4": stickers[3],
            "scraped_at": pd.Timestamp.utcnow()
        })

    return pd.DataFrame(rows)

In [203]:
def scrape_distinguishable_item_with_selenium(
    market_hash_name: str,
    pages: Union[int, str] = SELENIUM_PAGES_TO_SCRAPE,
    count: int = SELENIUM_RESULTS_PER_PAGE,
    headless: bool = False
) -> pd.DataFrame:

    driver = create_chrome_driver(headless=headless)
    all_rows = []

    try:
        base_url = build_listing_url(market_hash_name)

        first_url = f"{base_url}?start=0&count={count}"
        driver.get(first_url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.ID, "searchResultsRows"))
        )
        time.sleep(SELENIUM_PAGE_DELAY_SECONDS)

        first_html = driver.page_source
        total_listings = get_total_listings_from_html(first_html)

        if pages == "max" and total_listings:
            page_count = int(np.ceil(total_listings / count))
        elif pages == "max":
            page_count = 1
        else:
            page_count = int(pages)

        all_rows.append(parse_steam_listing_page(first_html, market_hash_name))

        for page in range(1, page_count):
            start = page * count
            url = f"{base_url}?start={start}&count={count}"
            driver.get(url)

            try:
                WebDriverWait(driver, 30).until(
                    lambda d: (
                        d.find_elements(By.ID, "searchResultsRows")
                        or d.find_elements(By.CLASS_NAME, "market_listing_row")
                        or "There are no listings" in d.page_source
                    )
                )
                time.sleep(SELENIUM_PAGE_DELAY_SECONDS)
            
                parsed = parse_steam_listing_page(driver.page_source, market_hash_name)
                all_rows.append(parsed)

            except TimeoutException:
                print(f"Timeout for {market_hash_name}, page {page + 1}. Skipping this page.")
                continue

    finally:
        driver.quit()

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True).drop_duplicates("listing_id")

In [204]:
skin_listings_df = scrape_distinguishable_item_with_selenium(
    DISTINGUISHABLE_ITEM,
    pages=SELENIUM_PAGES_TO_SCRAPE,
    count=SELENIUM_RESULTS_PER_PAGE,
    headless=False
)

skin_listings_df.head()

,market_hash_name,listing_id,asset_id,listing_url,price_with_fee,price_without_fee,total_price,float,pattern_template,inspect_link,sticker_1,sticker_2,sticker_3,sticker_4,scraped_at
0,AK-47 | Redline (Field-Tested),501731878520663201,51447514771,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.345456,236,B9A92A34326D06B8A1BE9923BB91BC89BD812B067A4CBA...,navi glitter,None,None,None,2026-05-11 23:43:18.694826+00:00
1,AK-47 | Redline (Field-Tested),505109578244534636,51449422853,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.367775,120,8C9C09447358338D948BAC168EA489BC88B406167D798F...,sig xyp9x,sig isak glitter,mouz,call your flashes paper,2026-05-11 23:43:18.695827+00:00
2,AK-47 | Redline (Field-Tested),493850860808337988,33852283261,https://steamcommunity.com/market/listings/730...,43.87,38.16,43.87,0.268939,14,FAEA0748797484E2FDDA60F8D2FFCAFEC2111E5C0EF9BA...,flux holo,None,None,None,2026-05-11 23:43:18.696829+00:00
3,AK-47 | Redline (Field-Tested),494976479274879462,51451508047,https://steamcommunity.com/market/listings/730...,44.22,38.46,44.22,0.311361,859,F4E43B1E0A214BF5ECF3D46EF6DCF1C4F0CC30210900F7...,sig donk champion holo,sig donk champion holo,sig donk champion holo,sig donk champion holo,2026-05-11 23:43:18.697829+00:00
4,AK-47 | Redline (Field-Tested),498354178748185051,51426362797,https://steamcommunity.com/market/listings/730...,44.51,38.71,44.51,0.322877,181,9989341219532698819EB9039BB19CA99DA12A390C6C9A...,astr holo,astr holo,astr holo,astr holo,2026-05-11 23:43:18.698831+00:00


## Feature engineering for distinguishable listings

The listing dataset is extended with price rank, float rank, a low-float flag and a simple potential-deal flag. The potential-deal flag identifies listings that are cheaper than the median listing and have a float below the selected threshold.

In [205]:
def engineer_listing_features(listings_df: pd.DataFrame, float_threshold: float = FLOAT_THRESHOLD) -> pd.DataFrame:
    out = listings_df.copy()

    if out.empty:
        return out

    price_column = "price_with_fee" if "price_with_fee" in out.columns else "price_without_fee"

    out["price_rank_pct"] = out[price_column].rank(pct=True, ascending=True)
    out["is_cheaper_than_median"] = out[price_column] < out[price_column].median()

    out["is_low_float"] = out["float"].notna() & (out["float"] < float_threshold)
    out["float_rank_pct"] = out["float"].rank(pct=True, ascending=True)

    out["has_stickers"] = out[["sticker_1", "sticker_2", "sticker_3", "sticker_4"]].notna().any(axis=1)
    out["sticker_count"] = out[["sticker_1", "sticker_2", "sticker_3", "sticker_4"]].notna().sum(axis=1)

    out["is_potential_deal"] = out["is_cheaper_than_median"] & out["is_low_float"]

    return out


skin_features_df = engineer_listing_features(skin_listings_df, FLOAT_THRESHOLD)
skin_features_df.head()

,market_hash_name,listing_id,asset_id,listing_url,price_with_fee,price_without_fee,total_price,float,pattern_template,inspect_link,...,sticker_3,sticker_4,scraped_at,price_rank_pct,is_cheaper_than_median,is_low_float,float_rank_pct,has_stickers,sticker_count,is_potential_deal
0,AK-47 | Redline (Field-Tested),501731878520663201,51447514771,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.345456,236,B9A92A34326D06B8A1BE9923BB91BC89BD812B067A4CBA...,...,None,None,2026-05-11 23:43:18.694826+00:00,0.050000,True,False,0.600000,True,1,False
1,AK-47 | Redline (Field-Tested),505109578244534636,51449422853,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.367775,120,8C9C09447358338D948BAC168EA489BC88B406167D798F...,...,mouz,call your flashes paper,2026-05-11 23:43:18.695827+00:00,0.050000,True,False,0.900000,True,4,False
2,AK-47 | Redline (Field-Tested),493850860808337988,33852283261,https://steamcommunity.com/market/listings/730...,43.87,38.16,43.87,0.268939,14,FAEA0748797484E2FDDA60F8D2FFCAFEC2111E5C0EF9BA...,...,None,None,2026-05-11 23:43:18.696829+00:00,0.100000,True,False,0.066667,True,1,False
3,AK-47 | Redline (Field-Tested),494976479274879462,51451508047,https://steamcommunity.com/market/listings/730...,44.22,38.46,44.22,0.311361,859,F4E43B1E0A214BF5ECF3D46EF6DCF1C4F0CC30210900F7...,...,sig donk champion holo,sig donk champion holo,2026-05-11 23:43:18.697829+00:00,0.133333,True,False,0.333333,True,4,False
4,AK-47 | Redline (Field-Tested),498354178748185051,51426362797,https://steamcommunity.com/market/listings/730...,44.51,38.71,44.51,0.322877,181,9989341219532698819EB9039BB19CA99DA12A390C6C9A...,...,astr holo,astr holo,2026-05-11 23:43:18.698831+00:00,0.166667,True,False,0.433333,True,4,False


In [206]:
skin_features_df.sort_values(
    ["is_potential_deal", "float", "price_with_fee"],
    ascending=[False, True, True]
).head(20)

,market_hash_name,listing_id,asset_id,listing_url,price_with_fee,price_without_fee,total_price,float,pattern_template,inspect_link,...,sticker_3,sticker_4,scraped_at,price_rank_pct,is_cheaper_than_median,is_low_float,float_rank_pct,has_stickers,sticker_count,is_potential_deal
27,AK-47 | Redline (Field-Tested),510739077837285862,50682877774,https://steamcommunity.com/market/listings/730...,63.65,55.36,63.65,0.211614,769,9C8C5232217B209D849BBC069EB499AC98A4507E7E6E9F...,...,sig im foil,sig im foil,2026-05-11 23:43:25.568900+00:00,0.816667,False,False,0.033333,True,4,False
2,AK-47 | Redline (Field-Tested),493850860808337988,33852283261,https://steamcommunity.com/market/listings/730...,43.87,38.16,43.87,0.268939,14,FAEA0748797484E2FDDA60F8D2FFCAFEC2111E5C0EF9BA...,...,None,None,2026-05-11 23:43:18.696829+00:00,0.100000,True,False,0.066667,True,1,False
24,AK-47 | Redline (Field-Tested),746041227273595451,45734810639,https://steamcommunity.com/market/listings/730...,63.87,55.55,63.87,0.280334,275,80900F2807302A819887A01A82A885B084B8680F3E7483...,...,None,None,2026-05-11 23:43:25.565083+00:00,0.950000,False,False,0.100000,False,0,False
21,AK-47 | Redline (Field-Tested),508486392210882710,50686111003,https://steamcommunity.com/market/listings/730...,63.25,55.00,63.25,0.282752,712,0E1E95D48CE7B20F16092E940C260B3E0A36D987CDFA0D...,...,None,None,2026-05-11 23:43:25.562081+00:00,0.700000,False,False,0.133333,True,2,False
25,AK-47 | Redline (Field-Tested),511864091785297977,51178721551,https://steamcommunity.com/market/listings/730...,63.78,55.47,63.78,0.287183,411,5040DFF2A583EE51485770CA527855605468FDC39CA453...,...,virtuspro,virtuspro,2026-05-11 23:43:25.566084+00:00,0.883333,False,False,0.166667,True,4,False
15,AK-47 | Redline (Field-Tested),634584189299036229,50529306644,https://steamcommunity.com/market/listings/730...,58.70,51.05,58.70,0.298252,270,0B1B9F9BAB95B70A130C2B9109230E3B0F33E7E3E9FF08...,...,None,None,2026-05-11 23:43:21.898690+00:00,0.650000,False,False,0.200000,True,1,False
8,AK-47 | Redline (Field-Tested),528753476544773277,51428789752,https://steamcommunity.com/market/listings/730...,45.44,39.52,45.44,0.303079,951,4A5AB2D1DE81F54B524D6AD048624F7A4E72EB90A6BE49...,...,None,None,2026-05-11 23:43:18.703834+00:00,0.266667,True,False,0.233333,False,0,False
19,AK-47 | Redline (Field-Tested),760679093991438165,49056046931,https://steamcommunity.com/market/listings/730...,57.88,50.34,57.88,0.303459,912,504083EE8F8FE651485770CA527855605468D0EEBDA453...,...,chin,None,2026-05-11 23:43:21.903694+00:00,0.566667,False,False,0.266667,True,3,False
11,AK-47 | Redline (Field-Tested),512990877832166555,51134744995,https://steamcommunity.com/market/listings/730...,57.22,49.77,57.22,0.308969,76,80902313793E3E819887A01A82A885B084B83362787483...,...,infinite diamond holo,infinite triangle holo,2026-05-11 23:43:21.894686+00:00,0.433333,True,False,0.300000,True,4,False
3,AK-47 | Redline (Field-Tested),494976479274879462,51451508047,https://steamcommunity.com/market/listings/730...,44.22,38.46,44.22,0.311361,859,F4E43B1E0A214BF5ECF3D46EF6DCF1C4F0CC30210900F7...,...,sig donk champion holo,sig donk champion holo,2026-05-11 23:43:18.697829+00:00,0.133333,True,False,0.333333,True,4,False


# Part 3: Scrapy as a scaling layer

Scrapy is used at the end of the workflow to scale the project from selected single items to multiple Steam Market item pages. It collects a lightweight catalog of item pages. This catalog can then be passed into the BeautifulSoup pipeline for indistinguishable items or the Selenium pipeline for distinguishable items.

In [207]:
SELECTED_ITEMS = [
    {"market_hash_name": "Revolution Case", "item_type": "indistinguishable"},
    {"market_hash_name": "Fracture Case", "item_type": "indistinguishable"},
    {"market_hash_name": "Snakebite Case", "item_type": "indistinguishable"},
    {"market_hash_name": "AK-47 | Redline (Field-Tested)", "item_type": "distinguishable"},
    {"market_hash_name": "M4A1-S | Cyrex (Field-Tested)", "item_type": "distinguishable"}
]

selected_items_df = pd.DataFrame(SELECTED_ITEMS)
selected_items_df["listing_url"] = selected_items_df["market_hash_name"].apply(build_listing_url)
selected_items_df

,market_hash_name,item_type,listing_url
0,Revolution Case,indistinguishable,https://steamcommunity.com/market/listings/730...
1,Fracture Case,indistinguishable,https://steamcommunity.com/market/listings/730...
2,Snakebite Case,indistinguishable,https://steamcommunity.com/market/listings/730...
3,AK-47 | Redline (Field-Tested),distinguishable,https://steamcommunity.com/market/listings/730...
4,M4A1-S | Cyrex (Field-Tested),distinguishable,https://steamcommunity.com/market/listings/730...


In [208]:
%%writefile steam_catalog_spider.py
import re
import scrapy
from urllib.parse import quote


APP_ID = 730


class SteamCatalogSpider(scrapy.Spider):
    name = "steam_catalog"

    custom_settings = {
        "DOWNLOAD_DELAY": 0.7,
        "CONCURRENT_REQUESTS": 2,
        "USER_AGENT": "Mozilla/5.0",
        "FEEDS": {
            "data/scrapy_catalog.json": {
                "format": "json",
                "overwrite": True
            }
        }
    }

    def __init__(self, items=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.items = items or ""

    def start_requests(self):
        for raw in self.items.split("||"):
            if not raw.strip():
                continue

            market_hash_name, item_type = raw.split("::")
            url = f"https://steamcommunity.com/market/listings/{APP_ID}/{quote(market_hash_name)}"

            yield scrapy.Request(
                url,
                callback=self.parse,
                meta={
                    "market_hash_name": market_hash_name,
                    "item_type": item_type,
                    "listing_url": url
                }
            )

    def parse(self, response):
        html = response.text
        match = re.search(r"Market_LoadOrderSpread\(\s*(\d+)\s*\)", html)
        item_nameid = int(match.group(1)) if match else None

        yield {
            "market_hash_name": response.meta["market_hash_name"],
            "item_type": response.meta["item_type"],
            "listing_url": response.meta["listing_url"],
            "page_title": response.css("title::text").get(),
            "item_nameid": item_nameid
        }

Overwriting steam_catalog_spider.py


In [209]:
scrapy_items_argument = "||".join(
    f"{row.market_hash_name}::{row.item_type}"
    for row in selected_items_df.itertuples(index=False)
)

scrapy_items_argument

'Revolution Case::indistinguishable||Fracture Case::indistinguishable||Snakebite Case::indistinguishable||AK-47 | Redline (Field-Tested)::distinguishable||M4A1-S | Cyrex (Field-Tested)::distinguishable'

In [210]:
!scrapy runspider steam_catalog_spider.py -a items="{scrapy_items_argument}"

2026-05-12 01:43:32 [scrapy.utils.log] INFO: Scrapy 2.15.2 started (bot: scrapybot)
2026-05-12 01:43:32 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.0',
 'libxml2': '2.11.9',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 '
           '64 bit (AMD64)]',
 'pyOpenSSL': '26.2.0 (OpenSSL 3.5.6 7 Apr 2026)',
 'cryptography': '46.0.7',
 'Platform': 'Windows-10-10.0.19045-SP0'}
2026-05-12 01:43:32 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-05-12 01:43:32 [asyncio] DEBUG: Using selector: SelectSelector
2026-05-12 01:43:32 [scrapy.addons] INFO: Enabled addons:
[]
2026-05-12 01:43:32 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-05-12 01:43:32 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.windows_events._WindowsSelectorEventLoop
2026-05-12 01:43:32 [scrapy.extensions.telnet] INFO: Telnet Password: 0

In [211]:
catalog_df = pd.read_json(DATA_DIR / "scrapy_catalog.json")
catalog_df.head()

,market_hash_name,item_type,listing_url,page_title,item_nameid
0,Revolution Case,indistinguishable,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for Revolut...,176358765
1,Fracture Case,indistinguishable,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for Fractur...,176185874
2,Snakebite Case,indistinguishable,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for Snakebi...,176240926
3,AK-47 | Redline (Field-Tested),distinguishable,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for AK-47 |...,7178002
4,M4A1-S | Cyrex (Field-Tested),distinguishable,https://steamcommunity.com/market/listings/730...,Steam Community Market :: Listings for M4A1-S ...,14962989


## Full-scale pipeline

The Scrapy catalog is split by item type. Indistinguishable items are processed with the aggregate BeautifulSoup pipeline. Distinguishable items are processed with the Selenium listing pipeline.

In [212]:
indistinguishable_catalog_df = catalog_df[catalog_df["item_type"].eq("indistinguishable")].copy()
distinguishable_catalog_df = catalog_df[catalog_df["item_type"].eq("distinguishable")].copy()

indistinguishable_catalog_df, distinguishable_catalog_df

(  market_hash_name          item_type  \
 0  Revolution Case  indistinguishable   
 1    Fracture Case  indistinguishable   
 2   Snakebite Case  indistinguishable   
 
                                          listing_url  \
 0  https://steamcommunity.com/market/listings/730...   
 1  https://steamcommunity.com/market/listings/730...   
 2  https://steamcommunity.com/market/listings/730...   
 
                                           page_title  item_nameid  
 0  Steam Community Market :: Listings for Revolut...    176358765  
 1  Steam Community Market :: Listings for Fractur...    176185874  
 2  Steam Community Market :: Listings for Snakebi...    176240926  ,
                  market_hash_name        item_type  \
 3  AK-47 | Redline (Field-Tested)  distinguishable   
 4   M4A1-S | Cyrex (Field-Tested)  distinguishable   
 
                                          listing_url  \
 3  https://steamcommunity.com/market/listings/730...   
 4  https://steamcommunity.com/market/list

In [213]:
all_history = []
all_orders = []

for row in indistinguishable_catalog_df.itertuples(index=False):
    result = scrape_indistinguishable_item(row.market_hash_name)
    features = engineer_case_features(result["history"], result["orders"])

    all_history.append(features["history"])
    all_orders.append(features["orders"])

multi_case_history_df = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
multi_case_orders_df = pd.concat(all_orders, ignore_index=True) if all_orders else pd.DataFrame()

multi_case_history_df.head()

,market_hash_name,date,median_price,volume,median_price_change_pct,volume_change_pct,median_price_7d_avg,volume_7d_avg
0,Revolution Case,2023-02-10 01:00:00+00:00,13.462,79764,NaN,NaN,13.4620,79764.000000
1,Revolution Case,2023-02-11 01:00:00+00:00,8.041,79886,-0.402689,0.001530,10.7515,79825.000000
2,Revolution Case,2023-02-12 01:00:00+00:00,6.889,78300,-0.143266,-0.019853,9.4640,79316.666667
3,Revolution Case,2023-02-13 01:00:00+00:00,6.960,58451,0.010306,-0.253499,8.8380,74100.250000
4,Revolution Case,2023-02-14 01:00:00+00:00,5.986,57769,-0.139943,-0.011668,8.2676,70834.000000


In [214]:
multi_case_orders_df.head()

,market_hash_name,side,price,quantity,description,scraped_at,best_buy_price,best_sell_price,buy_sell_spread_abs,buy_sell_spread_pct
0,Revolution Case,buy,0.49,19,19 buy orders at $0.49 or higher,2026-05-12 01:43:41.946283,0.49,0.5,0.01,0.02
1,Revolution Case,buy,0.46,1389,"1,389 buy orders at $0.46 or higher",2026-05-12 01:43:41.946283,0.49,0.5,0.01,0.02
2,Revolution Case,buy,0.43,16151,"16,151 buy orders at $0.43 or higher",2026-05-12 01:43:41.946283,0.49,0.5,0.01,0.02
3,Revolution Case,buy,0.42,22732,"22,732 buy orders at $0.42 or higher",2026-05-12 01:43:41.946283,0.49,0.5,0.01,0.02
4,Revolution Case,buy,0.41,56353,"56,353 buy orders at $0.41 or higher",2026-05-12 01:43:41.946283,0.49,0.5,0.01,0.02


In [215]:
all_skin_listings = []

for row in distinguishable_catalog_df.itertuples(index=False):
    listings = scrape_distinguishable_item_with_selenium(
        row.market_hash_name,
        pages=SELENIUM_PAGES_TO_SCRAPE,
        count=SELENIUM_RESULTS_PER_PAGE,
        headless=False
    )
    all_skin_listings.append(engineer_listing_features(listings, FLOAT_THRESHOLD))

multi_skin_listings_df = pd.concat(all_skin_listings, ignore_index=True) if all_skin_listings else pd.DataFrame()
multi_skin_listings_df.head()

,market_hash_name,listing_id,asset_id,listing_url,price_with_fee,price_without_fee,total_price,float,pattern_template,inspect_link,...,sticker_3,sticker_4,scraped_at,price_rank_pct,is_cheaper_than_median,is_low_float,float_rank_pct,has_stickers,sticker_count,is_potential_deal
0,AK-47 | Redline (Field-Tested),501731878520663201,51447514771,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.345456,236,B9A92A34326D06B8A1BE9923BB91BC89BD812B067A4CBA...,...,None,None,2026-05-11 23:44:02.172852+00:00,0.050000,True,False,0.600000,True,1,False
1,AK-47 | Redline (Field-Tested),505109578244534636,51449422853,https://steamcommunity.com/market/listings/730...,43.78,38.08,43.78,0.367775,120,8C9C09447358338D948BAC168EA489BC88B406167D798F...,...,mouz,call your flashes paper,2026-05-11 23:44:02.173853+00:00,0.050000,True,False,0.900000,True,4,False
2,AK-47 | Redline (Field-Tested),493850860808337988,33852283261,https://steamcommunity.com/market/listings/730...,43.87,38.16,43.87,0.268939,14,FAEA0748797484E2FDDA60F8D2FFCAFEC2111E5C0EF9BA...,...,None,None,2026-05-11 23:44:02.174854+00:00,0.100000,True,False,0.066667,True,1,False
3,AK-47 | Redline (Field-Tested),494976479274879462,51451508047,https://steamcommunity.com/market/listings/730...,44.22,38.46,44.22,0.311361,859,F4E43B1E0A214BF5ECF3D46EF6DCF1C4F0CC30210900F7...,...,sig donk champion holo,sig donk champion holo,2026-05-11 23:44:02.175854+00:00,0.133333,True,False,0.333333,True,4,False
4,AK-47 | Redline (Field-Tested),498354178748185051,51426362797,https://steamcommunity.com/market/listings/730...,44.51,38.71,44.51,0.322877,181,9989341219532698819EB9039BB19CA99DA12A390C6C9A...,...,astr holo,astr holo,2026-05-11 23:44:02.177856+00:00,0.166667,True,False,0.433333,True,4,False


# Saving final outputs

The final dataframes are saved as CSV files.

In [216]:
def slugify_name(name):
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


single_indistinguishable_dir = DATA_DIR / "example_indistinguishable_bs4" / slugify_name(INDISTINGUISHABLE_ITEM)
single_distinguishable_dir = DATA_DIR / "example_distinguishable_selenium" / slugify_name(DISTINGUISHABLE_ITEM)
multi_indistinguishable_dir = DATA_DIR / "indistinguishable_multi"
multi_distinguishable_dir = DATA_DIR / "distinguishable_multi"

for folder in [
    single_indistinguishable_dir,
    single_distinguishable_dir,
    multi_indistinguishable_dir,
    multi_distinguishable_dir
]:
    folder.mkdir(parents=True, exist_ok=True)


case_overview_df.to_csv(single_indistinguishable_dir / "overview.csv", index=False)
case_history_features_df.to_csv(single_indistinguishable_dir / "history_features.csv", index=False)
case_orders_features_df.to_csv(single_indistinguishable_dir / "orders_features.csv", index=False)

skin_features_df.to_csv(single_distinguishable_dir / "listing_features.csv", index=False)

multi_case_history_df.to_csv(multi_indistinguishable_dir / "history_features.csv", index=False)
multi_case_orders_df.to_csv(multi_indistinguishable_dir / "orders_features.csv", index=False)
multi_skin_listings_df.to_csv(multi_distinguishable_dir / "listing_features.csv", index=False)

sorted(DATA_DIR.rglob("*.csv"))

[WindowsPath('data/distinguishable_multi/listing_features.csv'),
 WindowsPath('data/example_distinguishable_selenium/ak_47_redline_field_tested/listing_features.csv'),
 WindowsPath('data/example_indistinguishable_bs4/revolution_case/history_features.csv'),
 WindowsPath('data/example_indistinguishable_bs4/revolution_case/orders_features.csv'),
 WindowsPath('data/example_indistinguishable_bs4/revolution_case/overview.csv'),
 WindowsPath('data/indistinguishable_multi/history_features.csv'),
 WindowsPath('data/indistinguishable_multi/orders_features.csv')]

# Summary

The notebook applies different scraping approaches to different market structures.

For indistinguishable items, the most useful data is aggregate market data, so the workflow uses `requests`, `BeautifulSoup` and regular expressions to obtain item identifiers, historical prices, volume and order book data.

For distinguishable items, the workflow uses Selenium to load listing pages and then parses embedded Steam asset data to extract listing-level attributes such as float value, pattern template and stickers.

Scrapy is used as a scaling layer that creates a catalog of multiple item pages, which is then passed to the appropriate downstream scraper depending on item type.